# 🧠 Semana 1: Prototipo Interactivo (Cerebro IA)

Este notebook me permite probar de forma interactiva toda la lógica que he desarrollado para la **Semana 1**. Aquí probaré el esquema de datos, la llamada al LLM para análisis de sentimiento, y el ruteo básico con LangGraph.

**LLM Seleccionado:** Cohere Command R (via `langchain-cohere`)

In [1]:
import os
from dotenv import load_dotenv

# Cargo las variables de entorno desde el archivo .env
load_dotenv(override=True)

# Verifico que la clave de Cohere esté disponible
api_key = os.getenv("COHERE_API_KEY")
if api_key:
    print(f"✅ COHERE_API_KEY cargada correctamente (termina en: ...{api_key[-4:]})") 
else:
    print("❌ COHERE_API_KEY no encontrada. Asegúrate de que el archivo .env tenga la variable COHERE_API_KEY definida.")

✅ COHERE_API_KEY cargada correctamente (termina en: ...4v4H)


### 1. Definición del Estado Global (AgentState)
Utilizo `Pydantic` para asegurar el tipado estricto de los datos. Estos esquemas son el contrato de datos de todo el pipeline.

In [2]:
from typing import List, Dict, Any
from pydantic import BaseModel, Field

class InteractionData(BaseModel):
    """Estructura de un mensaje individual de la comunidad."""
    autor: str = Field(description="Nombre del autor")
    canal: str = Field(description="Canal de origen")
    tipo: str = Field(description="Tipo de interacción")
    texto: str = Field(description="Contenido del mensaje")

class InteractionAnalysis(BaseModel):
    """Resultado del análisis cognitivo realizado por el LLM."""
    sentimiento: str = Field(description="Positivo, Negativo, Neutral o Altamente Positivo")
    temas: List[str] = Field(description="Temas extraídos del mensaje")
    puntuacion_relevancia: float = Field(description="Puntuación de 0.0 a 10.0")
    ruta_sugerida: str = Field(description="post_linkedin, faq o newsletter")

class AgentState(BaseModel):
    """Estado global del grafo. Transporta datos entre nodos."""
    interacciones: List[InteractionData] = Field(default_factory=list)
    analisis: Dict[int, InteractionAnalysis] = Field(default_factory=dict)
    errores: List[str] = Field(default_factory=list)

print("✅ Esquemas de datos (Pydantic) definidos correctamente.")

✅ Esquemas de datos (Pydantic) definidos correctamente.


### 2. Nodo de Análisis de Sentimiento (con Cohere)
He diseñado este nodo para que procese cada mensaje, llamando a **Cohere Command R** y obteniendo el output estructurado en JSON. Aplica manejo de errores robusto con `try/except`.

In [3]:
from langchain_cohere import ChatCohere
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

ANALYSIS_PROMPT = """
Eres un analista experto en comunidades digitales latinoamericanas.
Tu tarea es analizar el siguiente mensaje de un miembro de la comunidad y extraer la siguiente información.

1. Sentimiento general: puede ser Positivo, Altamente Positivo, Negativo o Neutral.
2. Temas principales: una lista de temas identificados (ej. Logros, Contratacion, Dudas tecnicas, Feedback, Spam).
3. Puntuacion de relevancia: un número decimal de 0.0 a 10.0. Un 10 representa un testimonio inspirador o una queja crítica muy valiosa.
4. Ruta sugerida: debe ser una de estas opciones -> post_linkedin, faq, newsletter.

Mensaje a analizar:
{mensaje}

Responde ÚNICAMENTE con el siguiente formato JSON exacto, sin texto adicional:
{format_instructions}
"""

def analyzer_node(state: AgentState) -> Dict[str, Any]:
    """Nodo de LangGraph que analiza el sentimiento y categoriza los mensajes de la comunidad."""
    print("🚀 Ejecutando analyzer_node con Cohere command-r-08-2024...")

    parser = JsonOutputParser(pydantic_object=InteractionAnalysis)

    prompt = ChatPromptTemplate.from_template(
        template=ANALYSIS_PROMPT,
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )

    llm = ChatCohere(
        model="command-r-08-2024",
        temperature=0,
        cohere_api_key=os.getenv("COHERE_API_KEY")
    )
    chain = prompt | llm | parser

    nuevos_analisis = {}
    errores = list(state.errores)

    for idx, msg in enumerate(state.interacciones):
        try:
            print(f"\n📩 [{idx+1}/{len(state.interacciones)}] Analizando mensaje de: '{msg.autor}'")
            resultado_json = chain.invoke({"mensaje": msg.texto})
            analisis = InteractionAnalysis(**resultado_json)
            nuevos_analisis[idx] = analisis
            print(f"   ✅ Sentimiento: {analisis.sentimiento} | Relevancia: {analisis.puntuacion_relevancia}/10 | Ruta: {analisis.ruta_sugerida}")
            print(f"   📌 Temas: {analisis.temas}")
        except Exception as e:
            print(f"   ❌ Error al procesar mensaje {idx+1}: {e}")
            errores.append(f"Mensaje {idx+1} ({msg.autor}): {str(e)}")

    print("\n✅ Nodo de análisis finalizado.")
    return {"analisis": nuevos_analisis, "errores": errores}

### 3. Prueba Rápida con Casos Reales del Proyecto
Introduzco tres casos dummy simulados del PDF (testimonio positivo, duda técnica y mensaje irrelevante) para probar el flujo completo del nodo.

In [4]:
# Defino el estado inicial de prueba con 3 casos de distinto tipo
estado_prueba = AgentState(
    interacciones=[
        InteractionData(
            autor="Mariana Souza",
            canal="#logros-y-empleos",
            tipo="testimonio",
            texto="Comunidad, quedé seleccionada para el puesto de Desarrolladora Junior de IA! Muy agradecida con la comunidad por todo el apoyo!"
        ),
        InteractionData(
            autor="Lucas Fernández",
            canal="#dudas-tecnicas",
            tipo="duda",
            texto="Hola, tengo un problema con la instalación de LangGraph, me arroja un error de dependencias con pydantic. ¿Alguien lo ha resuelto?"
        ),
        InteractionData(
            autor="Anónimo",
            canal="#general",
            tipo="spam",
            texto="jajaja que divertido el meme que vi hoy"
        )
    ]
)

# Ejecuto el nodo de análisis
resultado = analyzer_node(estado_prueba)

print("\n" + "="*50)
print("📊 RESUMEN FINAL DEL ESTADO")
print("="*50)
if resultado['errores']:
    print(f"\n⚠️ Errores registrados: {resultado['errores']}")
else:
    print(f"\n✅ Análisis completado sin errores. {len(resultado['analisis'])} mensajes procesados.")

🚀 Ejecutando analyzer_node con Cohere command-r-08-2024...

📩 [1/3] Analizando mensaje de: 'Mariana Souza'
   ✅ Sentimiento: Altamente Positivo | Relevancia: 9.5/10 | Ruta: post_linkedin
   📌 Temas: ['Logros', 'Contratacion']

📩 [2/3] Analizando mensaje de: 'Lucas Fernández'
   ✅ Sentimiento: Negativo | Relevancia: 7.5/10 | Ruta: faq
   📌 Temas: ['Dudas técnicas', 'Errores']

📩 [3/3] Analizando mensaje de: 'Anónimo'
   ✅ Sentimiento: Positivo | Relevancia: 6.0/10 | Ruta: post_linkedin
   📌 Temas: ['Humor', 'Entretenimiento']

✅ Nodo de análisis finalizado.

📊 RESUMEN FINAL DEL ESTADO

✅ Análisis completado sin errores. 3 mensajes procesados.


In [5]:
# PASO 3: Prueba del grafo LangGraph ensamblado desde producción (src/agents/graph.py)
import sys
import os
# Aseguramos que la ruta base del proyecto esté en el path para los imports de 'src'
if os.path.abspath('.') not in sys.path:
    sys.path.append(os.path.abspath('.'))

from src.agents.graph import app_graph
from src.agents.state import AgentState, InteractionData

# 1. Inicializar un estado de prueba
estado_prueba = AgentState(
    interacciones=[
        InteractionData(
            autor="Danny", 
            canal="#general", 
            tipo="duda", 
            texto="¿Cómo funciona la conexión entre LangGraph y Cohere?"
        )
    ]
)

# 2. Ejecutar el grafo compilado
print("🚀 Ejecutando el grafo de LangGraph desde producción...")
resultado = app_graph.invoke(estado_prueba)

# 3. Mostrar el resultado bonito
import pprint
pprint.pprint(resultado)


2026-09-20 15:29:37,730 - src.agents.graph - INFO - Ensamblando el grafo de LangGraph...
2026-09-20 15:29:37,758 - src.agents.graph - INFO - ✅ Grafo ensamblado y compilado correctamente con enrutamiento.
🚀 Ejecutando el grafo de LangGraph desde producción...
2026-09-20 15:29:37,764 - src.agents.nodes.analyzer_node - INFO - Iniciando nodo de análisis (analyzer_node)...
2026-09-20 15:29:37,821 - src.agents.nodes.analyzer_node - INFO - Analizando interacción 1/1 del autor 'Danny'...
2026-09-20 15:29:39,134 - src.agents.nodes.analyzer_node - INFO - ✅ Análisis completado para interacción 1. Sentimiento: Neutral
2026-09-20 15:29:39,135 - src.agents.nodes.analyzer_node - INFO - Finalizando nodo de análisis.
2026-09-20 15:29:39,136 - src.agents.nodes.router_node - INFO - 🔀 Iniciando enrutamiento condicional...
2026-09-20 15:29:39,137 - src.agents.nodes.router_node - INFO - Ruta sugerida por el LLM detectada: 'faq'
2026-09-20 15:29:39,137 - src.agents.nodes.router_node - INFO - ➡️ Bifurcando ha